# Host–Guest Committor: Model Validation

Checks stable-state separation, calibration against shooting outcomes, AIMMD loss behavior, and path crossing probabilities.

This notebook is independently runnable: it imports its dependencies, resolves
the project paths, loads the RPE trainset and trained committor model, creates
the `SystemVisualizer`, and restores or computes the model-output cache.


## 1. Imports and shared analysis helpers


In [ ]:
import os
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import aimmd
import aimmd.pytorch
import aimmd.pytorch.rcmodel
import aimmdTIS
from aimmdTIS import Tools as TAI_Tools
from aimmdTIS.visualization import SystemVisualizer



## 2. Project and model configuration


In [ ]:
DEFAULT_WORK_DIR = Path.cwd()
WORK_DIR = Path(os.environ.get("HG_COMMITTOR_WORK_DIR", DEFAULT_WORK_DIR)).expanduser().resolve()
ANALYSIS_DIR = WORK_DIR

if not ANALYSIS_DIR.is_dir():
    raise FileNotFoundError(
        f"Analysis directory not found: {ANALYSIS_DIR}. "
        "Set HG_COMMITTOR_WORK_DIR to the project root."
    )
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

from committor_analysis_helpers import (
    aimmd_softplus_loss_terms,
    aimmd_smoothness_loss_terms,
    conditional_permutation_importance,
    descriptor_names,
    descriptor_swap_loss_matrix,
    permutation_influence_correlation,
    plot_correlated_pair,
    plot_square_matrix,
    predict_q,
    run_descriptor_correlation_tools,
    sigmoid_stable,
    subsample_arrays,
    weighted_gradient_importance,
    weighted_permutation_importance,
)

TRAINSET_FILE = WORK_DIR / "Files" / "trainset_rpe.pkl"

MODEL_DIR = WORK_DIR / "Files"
MODEL_FILE = MODEL_DIR / "aimmd_store_s5em04.h5"
MODEL_KEY = "stage1_full_model_model_best"
OUTPUT_DIR = WORK_DIR / "Figures"
CACHE_DIR = MODEL_DIR / ".cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

for required_file in (TRAINSET_FILE, MODEL_FILE):
    if not required_file.is_file():
        raise FileNotFoundError(required_file)

DESCRIPTOR_LABELS = [
    "Host-Guest Distance",
    "Guest Orientation Angle",
    "Guest Position Angle",
    "Hydrogen Bonds",
    "Waters in Cavity",
    "Hydrophobic Contact Score",
    "Shared Waters",
]
DESCRIPTOR_UNITS = ["nm", "rad", "rad", "", "", "", ""]
DESC_MIN = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
DESC_MAX = np.array([1.0, np.pi, np.pi, 4.0, 10.0, 60.0, 10.0])
N_BINS_DIM = np.array([200, 200, 200, 6, 50, 200, 50])
STATE_A, STATE_B = 0.05, 1.0
Q_MIN, Q_MAX = -45.0, 15.0
N_DESC = len(DESCRIPTOR_LABELS)


## 3. Load the RPE trainset and committor model


In [ ]:
with TRAINSET_FILE.open("rb") as handle:
    trainset = pickle.load(handle)

required_keys = {"descriptors", "weights", "shot_results"}
missing_keys = required_keys.difference(trainset)
if missing_keys:
    raise KeyError(f"Trainset is missing keys: {sorted(missing_keys)}")

descriptors = np.asarray(trainset["descriptors"])
weights = np.asarray(trainset["weights"], dtype=float).reshape(-1)
shot_results = np.asarray(trainset["shot_results"])

if descriptors.ndim != 2 or descriptors.shape[1] != N_DESC:
    raise ValueError(f"Expected descriptors with shape (N, {N_DESC}), got {descriptors.shape}.")
if len(weights) != len(descriptors) or shot_results.shape != (len(descriptors), 2):
    raise ValueError("Trainset descriptors, weights, and shot results are not aligned.")
if np.any(weights < 0) or not np.all(np.isfinite(weights)):
    raise ValueError("Trainset weights must be finite and non-negative.")

positive_weights = weights[weights > 0]
if positive_weights.size == 0:
    raise ValueError("Trainset contains no positive weights.")
weights = weights / positive_weights.min()
trainset["weights"] = weights 

storage = aimmd.Storage(str(MODEL_FILE), mode="r")
if MODEL_KEY not in storage.rcmodels:
    raise KeyError(f"Model key {MODEL_KEY!r} not found; available: {list(storage.rcmodels)}")
model = storage.rcmodels[MODEL_KEY]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TAI_Tools.model_to(model, device)

print(f"Frames: {len(descriptors):,}; descriptors: {descriptors.shape[1]}")
print(f"Weight range: {weights.min():.3g} – {weights.max():.3g}")
print(f"Model: {MODEL_KEY} on {device}")


## 4. Create and initialize the `SystemVisualizer`


In [ ]:
standard_value = np.average(descriptors, axis=0).tolist()
vis = SystemVisualizer(
    temperature=1.0,
    dims_extent=[0.0, 1.2, 0.0, float(np.pi)],
    total_num_descriptors=N_DESC,
    standard_value=standard_value,
    descriptor_labels=DESCRIPTOR_LABELS,
    desc_min=DESC_MIN,
    desc_max=DESC_MAX,
)
vis.load_trainset(trainset)

print(f"Visualizer descriptors: {vis._desc.shape}")
print(f"Visualizer physical descriptors: {vis._desc_phys.shape}")


## 5. Restore or compute model outputs


In [ ]:
Q_CACHE_FILE = CACHE_DIR / "q_model_mbar.npy"
RECOMPUTE_Q_MODEL = False  # set to True after changing the model or trainset
if Q_CACHE_FILE.is_file() and not RECOMPUTE_Q_MODEL:
    q_model = np.load(Q_CACHE_FILE)
    if q_model.shape != (len(descriptors),):
        raise ValueError(
            f"Cached model output has shape {q_model.shape}; expected {(len(descriptors),)}. "
            "Delete the stale cache and rerun this cell."
        )
    p_B = sigmoid_stable(q_model)
    vis._model_output_cache[model] = (p_B, q_model)
    print(f"Loaded model output from {Q_CACHE_FILE}")
else:
    p_B, q_model = vis._model_output_rpe(model)
    np.save(Q_CACHE_FILE, q_model)
    print(f"Computed and cached model output at {Q_CACHE_FILE}")

print(f"q range: {q_model.min():.3f} – {q_model.max():.3f}")


## 6. Prepare frames with shooting outcomes


In [ ]:
rng = np.random.default_rng(123)
reduced_indices = rng.permutation(len(descriptors))[:100_000]
valid = np.sum(vis._shot, axis=1) > 0 
valid = valid & np.isin(np.arange(len(descriptors)), reduced_indices)
X_valid = np.asarray(vis._desc)[valid]
shots_valid = np.asarray(vis._shot)[valid]
weights_valid = np.asarray(vis._w)[valid]
q_valid = q_model[valid]
p_B_valid = sigmoid_stable(q_valid)
loss_valid = aimmd_softplus_loss_terms(q_valid, shots_valid, weights_valid)
loss_smoothness_valid = aimmd_smoothness_loss_terms(model,X_valid, weights=weights_valid)
print(f"Frames with shooting outcomes: {len(X_valid):,}")
print(f"sum of AIMMD loss: {np.sum(loss_valid):.6g}")


## 7. Stable-state separation and interfaces


In [ ]:
in_state_A = vis._desc_phys[:, 0] < STATE_A
in_state_B = vis._desc_phys[:, 0] > STATE_B
in_state = np.column_stack([in_state_A, in_state_B])

interfaces = aimmdTIS.Tools.check_interfaces(
    model,
    ("A", "B"),
    vis._desc,
    in_state=in_state,
    overlap=0.3,
)
print(f"State A frames: {in_state_A.sum():,}")
print(f"State B frames: {in_state_B.sum():,}")
print("Stable-state q ranges:\n", interfaces["min_max_stable_q"])
print("Forward interfaces:", interfaces["forward_interfaces"])
print("Backward interfaces:", interfaces["backward_interfaces"])


## 8. Along-q model diagnostics


In [ ]:
n_bins_q = 1000

# fig, axes = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"wspace": 0.2})
fig, axes = plt.subplots(1, 2, figsize=(12, 4))


# ── Left: individual loss / weight curves ─────────────────────────────────────
ax = axes[0]
vis.plot_loss_along_q(model,            ax=ax, n_bins=n_bins_q, color="black")
vis.plot_plnp_along_q(model,            ax=ax, n_bins=n_bins_q, color="steelblue")
vis.plot_loss_normalized_along_q(model, ax=ax, n_bins=n_bins_q, color="orange")
vis.plot_weight_along_q(model,          ax=ax, n_bins=n_bins_q, color="seagreen")
q_theory = np.linspace(Q_MIN, Q_MAX, 100)
p_B_theory = 1.0 / (1.0 + np.exp(-q_theory))
Information = -(p_B_theory * np.log(p_B_theory) + (1 - p_B_theory) * np.log(1 - p_B_theory))
ax.plot(q_theory, Information, color="darkblue", linestyle="--",lw=2, label=r"Ideal $I(q')$")


ax.set_yscale("log")
ax.set_xlim([Q_MIN, Q_MAX])
ax.set_xlabel(r"$q'$", fontsize=12)
ax.legend(loc="lower left", fontsize=9, frameon=False,ncol=3)
ax.grid(True, which="both", linestyle="--", alpha=0.5)
ax.set_xlabel(r"$q'$", fontsize=vis.plot_settings.fontsize)
# ax.set_title("Loss diagnostics along q", fontsize=12)
ax.set_ylim(bottom= 1/vis._w.sum())

# ── Right: frame-count distribution with twin axis ────────────────────────────
ax_right = axes[1]

edges, p_A, p_B = vis.plot_pA_pB_along_q(model, ax=ax_right, n_bins=n_bins_q)
# Theoretical curves
q_space = edges[1:]
ax_right.plot(q_space, 1 / (1 + np.exp( q_space)), label=r"$p_A$ theory", linestyle="--",lw=2, color="C0")
ax_right.plot(q_space, 1 / (1 + np.exp(-q_space)), label=r"$p_B$ theory", linestyle="--",lw=2, color="C1")

ax_right.set_yscale("log")
ax_right.set_xlim([Q_MIN, Q_MAX])
ax_right.set_xlabel(r"$q'$", fontsize=vis.plot_settings.fontsize)
ax_right.set_ylabel(r"$p_A$, $p_B$", fontsize=vis.plot_settings.fontsize)
ax_right.legend(loc="lower right", fontsize=9, frameon=False,ncol=2)
ax_right.grid(True, which="both", linestyle="--", alpha=0.5)
ax_right.set_ylim(bottom=1/vis._w.sum())

fig.tight_layout()
TAI_Tools.save_fig_pdf_and_png(fig, "HG_overview_q_chevron", output_path=OUTPUT_DIR)
plt.show()


fig, axes = plt.subplots(1, 2, figsize=(16, 5))
vis.plot_loss_along_q(model, ax=axes[0], n_bins=400, color="black")
vis.plot_plnp_along_q(model, ax=axes[0], n_bins=400, color="steelblue")
vis.plot_loss_normalized_along_q(model, ax=axes[0], n_bins=400, color="orange")
axes[0].set(yscale="log", xlim=(Q_MIN, Q_MAX), xlabel="model logit q")
axes[0].legend(frameon=False)

right = axes[1].twinx()
vis.plot_weight_along_q(model, ax=axes[1], n_bins=400, color="seagreen", density=True)
vis.plot_distribution_along_q(model, ax=right, n_bins=400, color="tomato", density=False)
axes[1].set(yscale="log", xlim=(Q_MIN, Q_MAX), xlabel="model logit q")
right.set_yscale("log")
fig.tight_layout()
TAI_Tools.save_fig_pdf_and_png(fig, "HG_committor_validation_along_q", output_path=OUTPUT_DIR)
plt.show()


## 9. Calibration against shooting outcomes


In [ ]:
q_edges = np.linspace(np.percentile(q_valid, 0.1), np.percentile(q_valid, 99.9), 1000)
bin_id = np.digitize(q_valid, q_edges) - 1
centers, empirical, predicted, support = [], [], [], []
shot_totals = shots_valid.sum(axis=1)

for index in range(len(q_edges) - 1):
    selected = bin_id == index
    if not np.any(selected):
        continue
    weighted_shots = weights_valid[selected] * shot_totals[selected]
    denominator = weighted_shots.sum()
    if denominator <= 0:
        continue
    centers.append(0.5 * (q_edges[index] + q_edges[index + 1]))
    empirical.append(np.sum(weights_valid[selected] * shots_valid[selected, 1]) / denominator)
    predicted.append(np.average(p_B_valid[selected], weights=weighted_shots))
    support.append(selected.sum())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(centers, empirical, "o", ms=4, label="shooting outcomes")
axes[0].plot(centers, predicted, "-", label="model prediction")
axes[0].set(xlabel="model logit q", ylabel=r"$p_B$", ylim=(-0.02, 1.02))
axes[0].legend(frameon=False)
axes[0].grid(alpha=0.2)

axes[1].scatter(predicted, empirical, c=np.log10(support), s=25, cmap="viridis")
axes[1].plot([0, 1], [0, 1], "--", color="black")
axes[1].set(xlabel="predicted pB", ylabel="empirical pB", xlim=(0, 1), ylim=(0, 1))
axes[1].set_aspect("equal")
fig.tight_layout()
TAI_Tools.save_fig_pdf_and_png(fig, "HG_committor_calibration", output_path=OUTPUT_DIR)
plt.show()


In [ ]:
print(np.sum(loss_smoothness_valid<10**(-1)))

## 10. Direct Committor Analysis Validation

In [ ]:
# Paths and settings
committor_dir = WORK_DIR / "Files"

descriptors_npy = committor_dir / "committor_analysis_descriptors_all_frames.npy"
shots_csv = committor_dir / "committor_analysis_all_frames.csv"
cache_file_committor_analysis = CACHE_DIR/ "committor_validation_stats_from_frames_.npz"

figure_name = "HG_committor_validation_committor_frames"

dq = 0.6
bins_q = np.arange(-6.0, 6+dq, dq)
q_lim = 4.5

# Load committor-analysis frame data and model
frame_data = aimmdTIS.diagnostics.load_committor_frame_data(
    descriptors_npy=descriptors_npy,
    shots_csv=shots_csv,
    bound_col="bound",
    unbound_col="unbound",
)

# Compute or load validation statistics cache
p_b_ref, q_ref = aimmdTIS.diagnostics.estimate_pb_from_shots(frame_data["shot_results"], eps=5e-4)
q_model_c_analysis = predict_q(model, frame_data["descriptors"])

stats = aimmdTIS.diagnostics.run_validation_with_cache(
    q_model=q_model_c_analysis,
    p_b_ref=p_b_ref,
    cache_file=cache_file_committor_analysis,
    force_recompute=True,
    bins_q=bins_q,
    eps=5e-4,
)

print("valid samples:", int(np.sum(stats["valid_mask"])))
print("RMSE q:", float(stats["rmse_q"][0]))
print("RMSE p_B:", float(stats["rmse_pb"][0]))

# Recreate the classic two-panel committor validation plot
fig, ax = aimmdTIS.plot_validation_panels(
    stats,
    lim_q=q_lim,
    title=f"Committor validation through direct committor-analysis ",
)

TAI_Tools.save_fig_pdf_and_png(fig, figure_name, output_path=OUTPUT_DIR)
plt.show()


## 11. Per-frame loss diagnostics


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------- data ----------

# ---------- selections ----------
sel_A = shots_valid[:, 0] == 2
sel_B = shots_valid[:, 1] == 2
sel_reactive = (shots_valid[:, 0] == 1) & (shots_valid[:, 1] == 1)
sel_in_state = X_valid[:, 0] < 0.05

finite = (
    np.isfinite(q_valid)
    & np.isfinite(loss_valid)
    & np.isfinite(loss_smoothness_valid)
    & np.isfinite(weights_valid)
)
    # ---------- top loss diagnostics ----------
top_loss = np.argsort(loss_valid[finite])[::-1][:10]
finite_indices = np.where(finite)[0]
top_loss_idx = finite_indices[top_loss]
# ---------- robust manual binning ----------
bin_width = 0.25

q_f = q_valid[finite]
loss_f = loss_valid[finite]
smooth_f = loss_smoothness_valid[finite]
w_f = weights_valid[finite]

q_min = np.floor(np.nanpercentile(q_f, 0.5) / bin_width) * bin_width
q_max = np.ceil(np.nanpercentile(q_f, 99.5) / bin_width) * bin_width
q_min = max(q_min, -50.0)
q_max = min(q_max, 20.0)

bin_id = np.floor((q_f - q_min) / bin_width).astype(int)
n_bins = int(np.ceil((q_max - q_min) / bin_width))

valid_bins = (bin_id >= 0) & (bin_id < n_bins)

bin_id = bin_id[valid_bins]
q_used = q_f[valid_bins]
loss_used = loss_f[valid_bins]
smooth_used = smooth_f[valid_bins]
w_used = w_f[valid_bins]

bin_centers = q_min + (np.arange(n_bins) + 0.5) * bin_width

hist_count = np.bincount(bin_id, minlength=n_bins)
hist_loss_sum = np.bincount(bin_id, weights=loss_used, minlength=n_bins)
hist_smooth_sum = np.bincount(bin_id, weights=smooth_used, minlength=n_bins)
hist_weight = np.bincount(bin_id, weights=np.sum(shots_valid[valid_bins], axis=1) * w_used, minlength=n_bins)
hist_weight_sum = np.bincount(bin_id, weights=w_used, minlength=n_bins)

mean_loss_per_frame = hist_loss_sum / np.maximum(hist_count, 1)
mean_smooth_per_frame = hist_smooth_sum / np.maximum(hist_count, 1)
weighted_mean_loss = hist_loss_sum / np.maximum(hist_weight, 1e-300)

nonempty = hist_count > 0

# ---------- figure ----------
fig, axes = plt.subplots(4, 1, figsize=(9, 13), sharex=True)

ax = axes[0]
ax.scatter(q_valid[sel_A & finite], loss_valid[sel_A & finite], s=8, alpha=0.35, label="2 shots to A")
ax.scatter(q_valid[sel_B & finite], loss_valid[sel_B & finite], s=8, alpha=0.35, label="2 shots to B")
ax.scatter(q_valid[sel_reactive & finite], loss_valid[sel_reactive & finite], s=12, alpha=0.60, label="1 shot A, 1 shot B")
ax.scatter(q_valid[sel_in_state & finite], loss_valid[sel_in_state & finite], s=5, alpha=0.50, label="in-state")
ax.scatter(q_valid[top_loss_idx], loss_valid[top_loss_idx], s=55, marker="x", linewidths=1.5, label="top 10 loss")
ax.set_yscale("log")
ax.set_ylabel("pointwise loss")
ax.axvline(0.0, linestyle="--", linewidth=1, alpha=0.6)
ax.legend(frameon=False, fontsize=8)
ax.grid(alpha=0.2)

ax = axes[1]
ax.scatter(q_valid[finite], loss_smoothness_valid[finite], s=5, alpha=0.15)
ax.plot(bin_centers[nonempty], mean_smooth_per_frame[nonempty], linewidth=2, label="binned mean smoothness")
ax.set_yscale("log")
ax.set_ylabel("smoothness loss")
ax.axvline(0.0, linestyle="--", linewidth=1, alpha=0.6)
ax.legend(frameon=False, fontsize=8)
ax.grid(alpha=0.2)

ax = axes[2]
ax.plot(bin_centers[nonempty], hist_loss_sum[nonempty], label="summed loss")
ax.plot(bin_centers[nonempty], mean_loss_per_frame[nonempty], label="mean loss per frame")
ax.plot(bin_centers[nonempty], weighted_mean_loss[nonempty], label="loss / weighted shot mass")
ax.set_yscale("log")
ax.set_ylabel("binned loss")
ax.axvline(0.0, linestyle="--", linewidth=1, alpha=0.6)
ax.legend(frameon=False, fontsize=8)
ax.grid(alpha=0.2)

ax = axes[3]
ax.plot(bin_centers[nonempty], hist_count[nonempty], label="frame count")
ax.plot(bin_centers[nonempty], hist_weight[nonempty], label="weighted shot mass")
ax.plot(bin_centers[nonempty], hist_weight_sum[nonempty], label="weight sum")
ax.set_yscale("log")
ax.set_xlabel("model logit committor q")
ax.set_ylabel("bin support")
ax.axvline(0.0, linestyle="--", linewidth=1, alpha=0.6)
ax.legend(frameon=False, fontsize=8)
ax.grid(alpha=0.2)
for a in axes:
    a.set_xlim(q_min-10, q_max+10)

fig.tight_layout()
plt.show()

In [ ]:
rng = np.random.default_rng(2026)
plot_size = min(100_000, len(q_valid))
plot_indices = rng.choice(len(q_valid), size=plot_size, replace=False)
q_plot = q_valid[plot_indices]
loss_plot = loss_valid[plot_indices]
shots_plot = shots_valid[plot_indices]

only_A = (shots_plot[:, 0] > 0) & (shots_plot[:, 1] == 0)
only_B = (shots_plot[:, 1] > 0) & (shots_plot[:, 0] == 0)
reactive = (shots_plot[:, 0] > 0) & (shots_plot[:, 1] > 0)

fig, axes = plt.subplots(2, 1, figsize=(10, 9), sharex=True)
for selected, label, color in [
    (only_A, "A-only shots", "seagreen"),
    (only_B, "B-only shots", "black"),
    (reactive, "shots to both states", "royalblue"),
]:
    axes[0].scatter(q_plot[selected], loss_plot[selected], s=5, alpha=0.25, label=label, color=color)
axes[0].set_yscale("log")
axes[0].set_ylabel("unweighted per-frame AIMMD loss")
axes[0].legend(frameon=False)

loss_sum, edges = np.histogram(q_valid, bins=200, weights=weights_valid * loss_valid)
weight_sum, _ = np.histogram(q_valid, bins=edges, weights=weights_valid)
mean_loss = np.divide(loss_sum, weight_sum, out=np.full_like(loss_sum, np.nan), where=weight_sum > 0)
axes[1].plot(0.5 * (edges[:-1] + edges[1:]), mean_loss)
axes[1].set(yscale="log", xlabel="model logit q", ylabel="weighted mean loss")
for ax in axes:
    ax.grid(alpha=0.2)
fig.tight_layout()
TAI_Tools.save_fig_pdf_and_png(fig, "HG_committor_loss_diagnostics", output_path=OUTPUT_DIR)
plt.show()


## 11. Forward path-crossing probability validation


In [ ]:
distance = np.asarray(vis._desc_phys[:, 0])
boundary = np.flatnonzero((distance < STATE_A) | (distance > STATE_B))
path_max_q, path_weights = [], []

for start, stop in zip(boundary[:-1], boundary[1:]):
    if stop - start <= 1 or distance[start] >= STATE_A:
        continue
    path_max_q.append(np.max(q_model[start:stop + 1]))
    path_weights.append(vis._w[start])

path_max_q = np.asarray(path_max_q)
path_weights = np.asarray(path_weights, dtype=float)
if len(path_max_q) == 0:
    raise ValueError("No paths originating in state A were reconstructed.")

selected_interfaces = np.linspace(-35, 0, 8)
fig, ax = plt.subplots(figsize=(12, 7))
colors = plt.cm.viridis(np.linspace(0.05, 0.95, len(selected_interfaces)))

for interface, color in zip(selected_interfaces, colors):
    crossed = path_max_q >= interface
    if not np.any(crossed):
        continue
    maxima = path_max_q[crossed]
    crossing_weights = path_weights[crossed]
    x = np.linspace(interface, max(interface + 1, np.percentile(maxima, 99.5)), 250)
    empirical = np.array([
        crossing_weights[maxima >= value].sum() / crossing_weights.sum()
        for value in x
    ])
    theoretical = sigmoid_stable(interface) / sigmoid_stable(x)
    ax.plot(x, empirical, color=color)
    ax.plot(x, theoretical, "--", color=color, alpha=0.8)

ax.plot([], [], color="black", label="RPE crossing probability")
ax.plot([], [], "--", color="black", label=r"$p_B(\lambda_i)/p_B(q)$")
ax.set(yscale="log", xlabel="model logit q", ylabel="crossing probability", ylim=(1e-4, 1.2))
ax.legend(frameon=False)
ax.grid(alpha=0.2)
fig.tight_layout()
TAI_Tools.save_fig_pdf_and_png(fig, "HG_crossing_probability_validation", output_path=OUTPUT_DIR)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

direction = "forward"
lw = 3
p_B, q_all = vis._model_output_rpe(model) 
if direction == "forward":
    interfaces = np.linspace(-35,5,10)
else:
    interfaces = np.linspace(-5,9,10)
fig, ax = plt.subplots(figsize=(16,8),dpi=300)
cmap = plt.get_cmap("tab20b")  # Built-in greyscale colormap
colors = [cmap(i / 10) for i in range(10)]  # 10 shades from dark to light
weights_total = vis._w/np.min(vis._w)
for i, interface in enumerate(interfaces):
    print(interface)
    # --- Build paths and per-path stats (unchanged logic, just numpy-ized) ---
    boundary_indices = np.where((vis._desc[:,0] > 1.0) | (vis._desc[:,0] < 0.05))[0]

    paths = []
    max_cv_values = []
    min_cv_values = []
    path_weight = []
    from_A = []
    from_B = []
    reactive = []
    crossed_interface_forward = []
    crossed_interface_backward = []

    for j in range(len(boundary_indices) - 1):
        start_idx = boundary_indices[j]
        end_idx   = boundary_indices[j + 1]
        if end_idx - start_idx > 1:  # avoid single-point/empty
            # slice covers a full path
            path = vis._desc[start_idx:end_idx + 1]
            paths.append(path)

            # mark if path starts in A
            from_A.append(vis._desc[start_idx, 0] < 0.05)
            from_B.append(vis._desc[start_idx, 0] > 1.0)
            reactive.append((np.any(vis._desc[start_idx:end_idx + 1,0] > 1.0) & np.any(vis._desc[start_idx:end_idx + 1,0] < 0.05)))

            # take the path's weight at the start (as in your code)
            path_weight.append(weights_total[start_idx])

            # path-wise extrema of q
            seg_q = q_all[start_idx:end_idx + 1]
            crossed_interface_forward.append(np.any((seg_q > interface))) 
            crossed_interface_backward.append(np.any((seg_q < interface)))
            max_cv_values.append(np.max(seg_q))
            min_cv_values.append(np.min(seg_q))

    from_A        = np.asarray(from_A, dtype=bool)
    from_B        = np.asarray(from_B, dtype=bool)   
    reactive      = np.asarray(reactive, dtype=bool)
    path_weight   = np.asarray(path_weight, dtype=np.float64)
    max_cv_values = np.asarray(max_cv_values, dtype=np.float64)
    min_cv_values = np.asarray(min_cv_values, dtype=np.float64)

    # --- Numerically robust weighted histogram and survival function ---
    bins = 1000
    rng  = (-50.0, 50.0)

    # Mask to select only paths coming from A
    if direction == "forward":
        mask = from_A & np.array(crossed_interface_forward)
    else:
        mask = from_B & np.array(crossed_interface_backward)

    # Rescale weights to avoid overflow (scale by max; any positive constant works)
    if np.any(mask):
        w = path_weight[mask]
        w_scale = np.max(w)
        if not np.isfinite(w_scale) or w_scale == 0:
            w_scale = 1.0
        w_safe = w / w_scale
    else:
        # No A-paths; keep a valid array
        w_safe = np.array([], dtype=np.float64)

    # Use density=False to get raw weighted counts (more stable for tails)
    if direction == "forward":
        counts, bin_edges = np.histogram(max_cv_values[mask], bins=bins, range=rng, weights=w_safe, density=False)
    else:
        counts, bin_edges = np.histogram(-min_cv_values[mask], bins=bins, range=rng, weights=w_safe, density=False)

    # Right-cumulative sum -> survival counts per lower bin edge
    # S_k = sum_{i >= k} counts_i / sum(counts)
    total = counts.sum()
    if total <= 0 or not np.isfinite(total):
        # Guard: nothing to plot
        xdata = bin_edges[1:]
        ydata = np.full_like(xdata, fill_value=np.finfo(float).tiny, dtype=np.float64)
    else:
        surv_counts = np.cumsum(counts[::-1])[::-1]
        ydata = surv_counts / total  # survival function P(max_q >= x_k)
        xdata = bin_edges[1:]        # right edges match the conventional plot location

    # Avoid zeros on log-scale
    tiny = np.finfo(float).tiny
    ydata = np.clip(ydata, tiny, 1.0)

    # --- (Optional) comparison curve if you want it ---
    q_linspace = np.linspace(interface, 13, 1000)
    if direction == "backward":
        q_linspace = np.linspace(-interface, 43, 1000)
    p_B = 1.0 / (1.0 + np.exp(- q_linspace))
    check_crossing = (p_B[0] / p_B)  # shape example you had; keep if useful
    # check_crossing = np.clip(check_crossing, tiny, None)
    # --- Plot ---
    if direction == "forward":
        ax.plot(xdata, ydata, color=colors[i%10], alpha=1, linewidth=lw, label=None if i>0 else r'RPE based $p_A(q|\lambda_{q_i})$')
    else:
        ax.plot(-xdata, ydata, color=colors[i%10], alpha=1, linewidth=lw, label=None if i>0 else r'RPE based $p_B(q|\lambda_{q_i})$')
    if direction == "forward":
        ax.plot(q_linspace, check_crossing, "--", c=colors[i%10], alpha=0.9, linewidth=lw, label=None if i>0 else r"$\frac{p_B(\lambda_{q_i})}{p_B(q)}$ theoretical $p_A(q|\lambda_{q_i})$")
    else:
        ax.plot(-q_linspace, check_crossing, "--", c=colors[i%10], alpha=0.9, linewidth=lw, label=None if i>0 else r"$\frac{p_A(\lambda_{q_i})}{p_A(q)}$ theoretical $p_B(q|\lambda_{q_i})$")
ax.set_ylim([1e-4,2])
ax.set_xlim([interfaces[0]-3, 10])
ax.set_yscale('log')
ax.set_xlabel(r'$q(x|\theta_{RPE_2})$', fontsize=20)
ax.set_ylabel('Crossing probability', fontsize=20)
ax.legend(fontsize=20)
plt.tight_layout()
TAI_Tools.save_fig_pdf_and_png(fig, "HG_crossing_probability_validation_newvis", output_path=OUTPUT_DIR)    

## 12 Combined publication figure: chevron plot, committor analysis result and crossing probability validation

In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(16, 8), constrained_layout=True)

# Overall layout
gs = fig.add_gridspec(
    nrows=2,
    ncols=3,
    height_ratios=[1, 1],
    width_ratios=[1.25, 1.25,  1],
)

# Top row
ax_chevron  = fig.add_subplot(gs[0, 0])
ax_pbpa = fig.add_subplot(gs[0, 1])

# Bottom left (stacked)
ax_q_committor_analysis    = fig.add_subplot(gs[0, 2])
ax_committor_analysis = fig.add_subplot(gs[1, 2])

# Bottom right (single large axis)
ax_crosprob = fig.add_subplot(gs[1, 0:2])

n_interfaces =10
fontsize_legends = 12
fontsize_labels = 12
plot_crossprob = True


# ── Left: individual loss / weight curves ─────────────────────────────────────
ax = ax_chevron
vis.plot_loss_along_q(model,            ax=ax, n_bins=n_bins_q, color="black")
vis.plot_plnp_along_q(model,            ax=ax, n_bins=n_bins_q, color="steelblue")
vis.plot_loss_normalized_along_q(model, ax=ax, n_bins=n_bins_q, color="orange")
vis.plot_weight_along_q(model,          ax=ax, n_bins=n_bins_q, color="seagreen")
q_theory = np.linspace(Q_MIN, Q_MAX, 100)
p_B_theory = 1.0 / (1.0 + np.exp(-q_theory))
Information = -(p_B_theory * np.log(p_B_theory) + (1 - p_B_theory) * np.log(1 - p_B_theory))
ax.plot(q_theory, Information, color="darkblue", linestyle="--",lw=2, label=r"Ideal $I(q')$")


ax.set_yscale("log")
ax.set_xlim([Q_MIN, Q_MAX])
ax.set_xlabel(r"$q'$", fontsize=fontsize_labels)
ax.legend(loc="lower left", fontsize=fontsize_legends, frameon=False,ncol=3)
# ax.grid(True, which="both", linestyle="--", alpha=0.5)
# ax.set_xlabel(r"$q'$", fontsize=vis.plot_settings.fontsize)
ax.set_xlabel(r"$q' =q(x|\theta)$", fontsize=fontsize_labels)

# ax.set_title("Loss diagnostics along q", fontsize=12)
ax.set_ylim(bottom= 1/(50*vis._w.sum()))

# ── Right: frame-count distribution with twin axis ────────────────────────────
ax = ax_pbpa

edges, p_A, p_B = vis.plot_pA_pB_along_q(model, ax=ax, n_bins=n_bins_q)
# Theoretical curves
q_space = edges[1:]
ax.plot(q_space, 1 / (1 + np.exp( q_space)), label=r"$p_A$ theory", linestyle="--",lw=2, color="C0")
ax.plot(q_space, 1 / (1 + np.exp(-q_space)), label=r"$p_B$ theory", linestyle="--",lw=2, color="C1")

ax.set_yscale("log")
ax.set_xlim([Q_MIN, Q_MAX])
# ax.set_xlabel(r"$q'$", fontsize=vis.plot_settings.fontsize)
ax.set_xlabel(r"$q' =q(x|\theta)$", fontsize=fontsize_labels)

 
ax.set_ylabel(r"$p_A$, $p_B$", fontsize=fontsize_labels)
ax.legend(loc="lower right", fontsize=fontsize_legends, frameon=False,ncol=2)
# ax.grid(True, which="both", linestyle="--", alpha=0.5)
ax.set_ylim(bottom=1/(50*vis._w.sum()))


# Recreate the classic two-panel committor validation plot
_, ax = aimmdTIS.plot_validation_panels(
    stats,
    lim_q=q_lim,
    title=None,
    ax=[ax_q_committor_analysis, ax_committor_analysis],
    fig=fig
)

# q_linspace = np.linspace(-q_lim, q_lim, 1000)
# pb_linspace = 1.0 / (1.0 + np.exp(-q_linspace))
# pb_min_fill = 1.0 / (1.0 + np.exp(-q_linspace+1))
# pb_max_fill = 1.0 / (1.0 + np.exp(-q_linspace-1))
# 

# ax_committor_analysis.fill_between(pb_linspace, pb_min_fill, pb_max_fill, color="lightgray", alpha=0.5, label=r"$\pm 1$ logit unit")
for a in [ax_q_committor_analysis, ax_committor_analysis]:
    a.legend(loc="lower right", fontsize=fontsize_legends, frameon=False)

    a.grid(False)


direction = "forward"
lw = 3
p_B, q_all = vis._model_output_rpe(model) 
if direction == "forward":
    interfaces = np.linspace(-35,5,n_interfaces)
else:
    interfaces = np.linspace(-5,9,10)
ax = ax_crosprob
if plot_crossprob:
    cmap = plt.get_cmap("tab20b")  # Built-in greyscale colormap
    colors = [cmap(i / 10) for i in range(10)]  # 10 shades from dark to light
    weights_total = vis._w/np.min(vis._w)
    for i, interface in enumerate(interfaces):
        print(interface)
        # --- Build paths and per-path stats (unchanged logic, just numpy-ized) ---
        boundary_indices = np.where((vis._desc[:,0] > 1.0) | (vis._desc[:,0] < 0.05))[0]

        paths = []
        max_cv_values = []
        min_cv_values = []
        path_weight = []
        from_A = []
        from_B = []
        reactive = []
        crossed_interface_forward = []
        crossed_interface_backward = []

        for j in range(len(boundary_indices) - 1):
            start_idx = boundary_indices[j]
            end_idx   = boundary_indices[j + 1]
            if end_idx - start_idx > 1:  # avoid single-point/empty
                # slice covers a full path
                path = vis._desc[start_idx:end_idx + 1]
                paths.append(path)
 
                # mark if path starts in A
                from_A.append(vis._desc[start_idx, 0] < 0.05)
                from_B.append(vis._desc[start_idx, 0] > 1.0)
                reactive.append((np.any(vis._desc[start_idx:end_idx + 1,0] > 1.0) & np.any(vis._desc[start_idx:end_idx + 1,0] < 0.05)))

                # take the path's weight at the start (as in your code)
                path_weight.append(weights_total[start_idx])

                # path-wise extrema of q
                seg_q = q_all[start_idx:end_idx + 1]
                crossed_interface_forward.append(np.any((seg_q > interface))) 
                crossed_interface_backward.append(np.any((seg_q < interface)))
                max_cv_values.append(np.max(seg_q))
                min_cv_values.append(np.min(seg_q))

        from_A        = np.asarray(from_A, dtype=bool)
        from_B        = np.asarray(from_B, dtype=bool)   
        reactive      = np.asarray(reactive, dtype=bool)
        path_weight   = np.asarray(path_weight, dtype=np.float64)
        max_cv_values = np.asarray(max_cv_values, dtype=np.float64)
        min_cv_values = np.asarray(min_cv_values, dtype=np.float64)

        # --- Numerically robust weighted histogram and survival function ---
        bins = 1000
        rng  = (-50.0, 50.0)

        # Mask to select only paths coming from A
        if direction == "forward":
            mask = from_A & np.array(crossed_interface_forward)
        else:
            mask = from_B & np.array(crossed_interface_backward)

        # Rescale weights to avoid overflow (scale by max; any positive constant works)
        if np.any(mask):
            w = path_weight[mask]
            w_scale = np.max(w)
            if not np.isfinite(w_scale) or w_scale == 0:
                w_scale = 1.0
            w_safe = w / w_scale
        else:
            # No A-paths; keep a valid array
            w_safe = np.array([], dtype=np.float64)

        # Use density=False to get raw weighted counts (more stable for tails)
        if direction == "forward":
            counts, bin_edges = np.histogram(max_cv_values[mask], bins=bins, range=rng, weights=w_safe, density=False)
        else:
            counts, bin_edges = np.histogram(-min_cv_values[mask], bins=bins, range=rng, weights=w_safe, density=False)

        # Right-cumulative sum -> survival counts per lower bin edge
        # S_k = sum_{i >= k} counts_i / sum(counts)
        total = counts.sum()
        if total <= 0 or not np.isfinite(total):
            # Guard: nothing to plot
            xdata = bin_edges[1:]
            ydata = np.full_like(xdata, fill_value=np.finfo(float).tiny, dtype=np.float64)
        else:
            surv_counts = np.cumsum(counts[::-1])[::-1]
            ydata = surv_counts / total  # survival function P(max_q >= x_k)
            xdata = bin_edges[1:]        # right edges match the conventional plot location

        # Avoid zeros on log-scale
        tiny = np.finfo(float).tiny
        ydata = np.clip(ydata, tiny, 1.0)

        # --- (Optional) comparison curve if you want it ---
        q_linspace = np.linspace(interface, 13, 1000)
        if direction == "backward":
            q_linspace = np.linspace(-interface, 43, 1000)
        p_B = 1.0 / (1.0 + np.exp(- q_linspace))
        check_crossing = (p_B[0] / p_B)  # shape example you had; keep if useful
        # check_crossing = np.clip(check_crossing, tiny, None)
        # --- Plot ---
        if direction == "forward":
            ax.plot(xdata, ydata, color=colors[i%10], alpha=1, linewidth=lw, label=None if i>0 else r'RPE based $p_A(q|\lambda_{q_i})$')
        else:
            ax.plot(-xdata, ydata, color=colors[i%10], alpha=1, linewidth=lw, label=None if i>0 else r'RPE based $p_B(q|\lambda_{q_i})$')
        if direction == "forward":
            ax.plot(q_linspace, check_crossing, "--", c=colors[i%10], alpha=0.9, linewidth=lw, label=None if i>0 else r"$\frac{p_B(\lambda_{q_i})}{p_B(q)}$ theoretical $p_A(q|\lambda_{q_i})$")
        else:
            ax.plot(-q_linspace, check_crossing, "--", c=colors[i%10], alpha=0.9, linewidth=lw, label=None if i>0 else r"$\frac{p_A(\lambda_{q_i})}{p_A(q)}$ theoretical $p_B(q|\lambda_{q_i})$")
ax.set_ylim([1e-4,2])
ax.set_xlim([interfaces[0]-3, 10])
ax.set_yscale('log')
ax.set_xlabel(r'$q(x|\theta)$', fontsize=fontsize_labels)
ax.set_ylabel('Crossing probability', fontsize=fontsize_labels)
ax.legend(fontsize=fontsize_legends, frameon=False)

from matplotlib.transforms import ScaledTranslation

labels = ["A", "B", "C","E", "D", ]

offset_x = -25  # pixels
offset_y = 15   # pixels

for i, ax in enumerate([ax_chevron, ax_pbpa, ax_q_committor_analysis,
                        ax_committor_analysis, ax_crosprob]):

    trans = ax.transAxes + ScaledTranslation(
        offset_x / fig.dpi,
        offset_y / fig.dpi,
        fig.dpi_scale_trans
    )

    ax.text(
        0, 1,
        labels[i],
        transform=trans,
        fontsize=20,
        fontweight="bold",
        va="bottom",
        ha="right",
    )

fig.tight_layout()

TAI_Tools.save_fig_pdf_and_png(fig, "HG_validation_overview", output_path=OUTPUT_DIR)

